# Análise exploratória de vendas: produtos, categorias e evolução

**Problema.** Um e-commerce fictício quer entender **como as vendas evoluíram** em 2023 e 2024, **quais produtos e categorias** sustentam a receita e **onde e por qual canal** ela acontece, para orientar estoque, marketing e metas.

**Dados.** ⚠️ A base é **sintética** (gerada por `src/gerar_dados.py` com semente fixa): 6.655 vendas fictícias de 2023 a 2024. Só a sazonalidade mensal foi calibrada com dados reais do IBGE. Os números servem para demonstrar o método de análise, **não** descrevem uma empresa real.

**Roteiro**
1. Carregar e conferir a qualidade da base
2. Evolução das vendas no tempo
3. Categorias e produtos
4. Regiões e canais
5. Sazonalidade
6. Conferência SQL × pandas
7. Indicadores e conclusões

In [1]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Os scripts do projeto ficam em src/ e são importados como módulos
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import analise
import carregar_sqlite
import gerar_dados

pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
em_mil = mticker.FuncFormatter(lambda x, _: f"{x / 1000:.0f}k")  # 250000 -> "250k" nos eixos

# Gera a base e o banco SQLite se ainda não existirem (mesma semente = mesmos dados)
if not analise.CSV_PATH.exists():
    gerar_dados.main()
if not (RAIZ / "data" / "vendas.db").exists():
    carregar_sqlite.main()

## 1. Carregar e conferir a qualidade da base

In [2]:
df = analise.carregar_dados()   # lê data/vendas.csv e cria as colunas ano e mes
print(f"{len(df)} vendas de {df['data'].min():%d/%m/%Y} a {df['data'].max():%d/%m/%Y}")
df.head()

6655 vendas de 01/01/2023 a 31/12/2024


,id_venda,data,categoria,produto,quantidade,preco_unitario,receita,regiao,canal,ano,mes
0,1,2023-01-01,Eletrônicos,Smartwatch,2,"1,223.05","2,446.10",Sudeste,Online,2023,2023-01
1,2,2023-01-01,Eletrônicos,Carregador,2,732.64,"1,465.28",Nordeste,Loja Física,2023,2023-01
2,3,2023-01-01,Livros,Técnico,2,104.42,208.84,Nordeste,Online,2023,2023-01
3,4,2023-01-01,Moda,Tênis,1,192.08,192.08,Nordeste,Loja Física,2023,2023-01
4,5,2023-01-01,Moda,Boné,1,123.85,123.85,Sul,Online,2023,2023-01


In [3]:
# Tipos, valores ausentes e valores distintos de cada coluna
pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(), "distintos": df.nunique()})

,tipo,nulos,distintos
id_venda,int64,0,6655
data,datetime64[us],0,731
categoria,str,0,5
produto,str,0,20
quantidade,int64,0,3
preco_unitario,float64,0,6233
receita,float64,0,6280
regiao,str,0,5
canal,str,0,2
ano,int32,0,2


In [4]:
# Checagens de consistência antes de analisar
checagens = {
    "ids de venda únicos": df["id_venda"].is_unique,
    "sem linhas duplicadas": not df.duplicated().any(),
    "receita = quantidade x preço": ((df["quantidade"] * df["preco_unitario"]).round(2) - df["receita"]).abs().max() < 0.01,
    "quantidades e preços positivos": bool((df["quantidade"] > 0).all() and (df["preco_unitario"] > 0).all()),
    "24 meses sem buracos": df["mes"].nunique() == 24,
}
pd.Series(checagens, name="ok")

ids de venda únicos               True
sem linhas duplicadas             True
receita = quantidade x preço      True
quantidades e preços positivos    True
24 meses sem buracos              True
Name: ok, dtype: bool

In [5]:
df[["quantidade", "preco_unitario", "receita"]].describe()

,quantidade,preco_unitario,receita
count,"6,655.00","6,655.00","6,655.00"
mean,1.26,530.29,665.59
std,0.55,600.37,861.04
min,1.00,20.00,20.00
25%,1.00,109.96,124.69
50%,1.00,253.66,309.14
75%,1.00,792.76,909.18
max,3.00,"3,500.00","10,500.00"


A base está consistente: nenhum valor ausente, nenhuma duplicata, receita sempre igual a quantidade × preço e todos os 24 meses presentes. A **receita por venda é muito assimétrica**: a mediana fica bem abaixo da média, porque poucas vendas de itens caros (eletrônicos, bicicletas) puxam a média para cima. Por isso, ao longo da análise, olhamos **receita total** e **número de vendas** separadamente, e não só o ticket médio.

## 2. Evolução das vendas no tempo

In [6]:
mensal = df.groupby("mes").agg(vendas=("id_venda", "size"), receita=("receita", "sum"))
mensal["ticket_medio"] = mensal["receita"] / mensal["vendas"]

fig, ax = plt.subplots(figsize=(11, 4))
mensal["receita"].plot(ax=ax, marker="o", color="#2563eb")
ax.yaxis.set_major_formatter(em_mil)
ax.set_title("Receita mensal")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()

In [7]:
anual = df.groupby("ano").agg(vendas=("id_venda", "size"), receita=("receita", "sum"), ticket_medio=("receita", "mean"))
anual.loc["variação %"] = (anual.loc[2024] / anual.loc[2023] - 1) * 100
anual

C:\Users\arthu\AppData\Local\Temp\ipykernel_22812\3968954600.py:2: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype int64 cannot hold the value being set, so the result has dtype float64. Cast the object to float64 before this operation to retain the current behavior.
  anual.loc["variação %"] = (anual.loc[2024] / anual.loc[2023] - 1) * 100


,vendas,receita,ticket_medio
ano,,,
2023,"3,075.00","2,014,881.46",655.25
2024,"3,580.00","2,414,610.47",674.47
variação %,16.42,19.84,2.93


A receita cresceu **19,8%** de 2023 para 2024. A decomposição mostra **de onde veio** esse crescimento: o **número de vendas subiu 16,4%** e o **ticket médio só 2,9%**. Ou seja, a empresa cresceu principalmente **vendendo mais vezes**, não vendendo mais caro. Dezembro é o pico nos dois anos.

## 3. Categorias e produtos

In [8]:
por_categoria = df.pivot_table(index="categoria", columns="ano", values="receita", aggfunc="sum")
por_categoria["crescimento %"] = (por_categoria[2024] / por_categoria[2023] - 1) * 100
por_categoria["% da receita"] = (por_categoria[2023] + por_categoria[2024]) / df["receita"].sum() * 100
por_categoria.sort_values("% da receita", ascending=False)

ano,2023,2024,crescimento %,% da receita
categoria,,,,
Eletrônicos,"974,514.65","1,203,534.36",23.50,49.17
Esporte,"672,818.42","784,281.20",16.57,32.90
Casa,"172,934.63","203,245.83",17.53,8.49
Moda,"142,572.01","167,255.03",17.31,6.99
Livros,"52,041.75","56,294.05",8.17,2.45


In [9]:
perfil = df.groupby("categoria").agg(vendas=("id_venda", "size"), unidades=("quantidade", "sum"),
                                     ticket_medio=("receita", "mean"))
perfil.sort_values("ticket_medio", ascending=False)

,vendas,unidades,ticket_medio
categoria,,,
Eletrônicos,1325,1640,"1,643.81"
Esporte,1330,1698,"1,095.56"
Casa,1281,1600,293.66
Moda,1374,1757,225.49
Livros,1345,1693,80.55


In [10]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
por_categoria["% da receita"].sort_values().plot.barh(ax=eixos[0], color="#059669", title="% da receita total")
perfil["vendas"].sort_values().plot.barh(ax=eixos[1], color="#9ca3af", title="Número de vendas")
for eixo in eixos:
    eixo.set_ylabel("")
plt.tight_layout()

As duas tabelas juntas contam a história: **todas as categorias vendem quase o mesmo número de vezes** (entre 1.281 e 1.374 vendas), mas o **ticket médio** varia de R$ 80 (Livros) a R$ 1.644 (Eletrônicos). É o preço, e não a frequência, que faz **Eletrônicos responder por quase metade da receita** e Livros por pouco mais de 2%. Eletrônicos também foi a categoria que **mais cresceu** (+23,5%), enquanto Livros cresceu bem menos (+8,2%).

In [11]:
top_produtos = (df.groupby(["produto", "categoria"])
                  .agg(vendas=("id_venda", "size"), receita=("receita", "sum"))
                  .sort_values("receita", ascending=False))
top_produtos["% acumulado"] = top_produtos["receita"].cumsum() / top_produtos["receita"].sum() * 100
top_produtos.head(10)

,,vendas,receita,% acumulado
produto,categoria,,,
Carregador,Eletrônicos,345,"564,652.37",12.75
Smartphone,Eletrônicos,330,"548,155.81",25.12
Fone de Ouvido,Eletrônicos,315,"538,822.95",37.29
Smartwatch,Eletrônicos,335,"526,417.88",49.17
Bola,Esporte,346,"383,770.72",57.84
Tapete de Yoga,Esporte,325,"364,988.64",66.08
Halteres,Esporte,345,"361,769.02",74.24
Bicicleta,Esporte,314,"346,571.24",82.07
Tapete,Casa,326,"99,350.73",84.31


Os **4 primeiros produtos são todos eletrônicos** e, juntos, somam cerca de metade da receita. Logo atrás vêm itens de Esporte. Há uma concentração forte: uma ruptura de estoque em eletrônicos teria impacto muito maior que em qualquer outra linha.

## 4. Regiões e canais

In [12]:
por_regiao = df.groupby("regiao").agg(vendas=("id_venda", "size"), receita=("receita", "sum"),
                                      ticket_medio=("receita", "mean"))
por_regiao["% da receita"] = por_regiao["receita"] / por_regiao["receita"].sum() * 100
por_regiao.sort_values("receita", ascending=False)

,vendas,receita,ticket_medio,% da receita
regiao,,,,
Sudeste,3068,"2,040,774.99",665.18,46.07
Nordeste,1450,"994,066.53",685.56,22.44
Sul,1203,"766,875.88",637.47,17.31
Centro-Oeste,559,"387,689.92",693.54,8.75
Norte,375,"240,084.61",640.23,5.42


In [13]:
canal_ano = df.pivot_table(index="ano", columns="canal", values="receita", aggfunc="sum")
canal_ano["% Online"] = canal_ano["Online"] / (canal_ano["Online"] + canal_ano["Loja Física"]) * 100

ax = canal_ano[["Online", "Loja Física"]].plot.bar(figsize=(7, 4), color=["#2563eb", "#9ca3af"], rot=0)
ax.yaxis.set_major_formatter(em_mil)
ax.set_title("Receita por canal e ano")
ax.set_xlabel("")
plt.tight_layout()
canal_ano

canal,Loja Física,Online,% Online
ano,,,
2023,"898,551.84","1,116,329.62",55.40
2024,"902,180.53","1,512,429.94",62.64


- O **Sudeste concentra cerca de 46%** da receita. O ticket médio é parecido entre as regiões, então a diferença vem do **volume de vendas**, que segue os pesos populacionais usados na geração dos dados.
- O **Online** passou de 55,4% para **62,6%** da receita. A loja física ficou praticamente estável, então **quase todo o crescimento de 2024 veio do canal Online**.

## 5. Sazonalidade

In [14]:
receita_mes_do_ano = df.groupby([df["data"].dt.month, "ano"])["receita"].sum().groupby(level=0).mean()
ax = receita_mes_do_ano.plot.bar(figsize=(9, 3.5), color=["#d97706" if m in (11, 12) else "#2563eb"
                                                          for m in receita_mes_do_ano.index], rot=0)
ax.yaxis.set_major_formatter(em_mil)
ax.set_title("Receita média por mês do ano (laranja = Nov/Dez)")
ax.set_xlabel("mês")
plt.tight_layout()

In [15]:
metricas = analise.calcular_metricas(df)   # mesma função usada pelo relatório e testada no pytest
efeito = metricas["media_mensal_alta_temporada"] / metricas["media_mensal_resto"]
print(f"Receita média mensal em Nov/Dez: R$ {metricas['media_mensal_alta_temporada']:,.2f}")
print(f"Receita média nos demais meses:  R$ {metricas['media_mensal_resto']:,.2f}")
print(f"Efeito de alta temporada: {efeito:.2f}x")

Receita média mensal em Nov/Dez: R$ 222,207.11
Receita média nos demais meses:  R$ 177,033.17
Efeito de alta temporada: 1.26x


Novembro e dezembro vendem **1,26x** mais que a média dos outros meses, com **dezembro como pico**; os meses mais fracos são **abril e fevereiro**. Esse formato vem da sazonalidade **real** do varejo brasileiro (IBGE) usada para calibrar o gerador (no IBGE, fevereiro é o mais fraco; aqui abril fica um pouco abaixo por variação aleatória da amostra). O efeito medido aqui é um pouco maior que o do IBGE (1,19x) porque a receita também cresce ao longo do tempo, e novembro/dezembro ficam no fim de cada ano.

## 6. Conferência SQL × pandas

As consultas de `sql/consultas.sql` rodam sobre o banco SQLite. Aqui a mesma pergunta é respondida pelas duas ferramentas para garantir que **os números batem**:

In [16]:
conexao = sqlite3.connect(RAIZ / "data" / "vendas.db")

sql_categoria = pd.read_sql_query("""
    SELECT categoria,
           ROUND(SUM(receita), 2) AS receita_total,
           ROUND(100.0 * SUM(receita) / (SELECT SUM(receita) FROM vendas), 2) AS pct_do_total
    FROM vendas
    GROUP BY categoria
    ORDER BY receita_total DESC
""", conexao).set_index("categoria")

pandas_categoria = df.groupby("categoria")["receita"].sum().round(2)
sql_categoria["receita_pandas"] = pandas_categoria
sql_categoria["bate?"] = (sql_categoria["receita_total"] - sql_categoria["receita_pandas"]).abs() < 0.01
sql_categoria

,receita_total,pct_do_total,receita_pandas,bate?
categoria,,,,
Eletrônicos,"2,178,049.01",49.17,"2,178,049.01",True
Esporte,"1,457,099.62",32.90,"1,457,099.62",True
Casa,"376,180.46",8.49,"376,180.46",True
Moda,"309,827.04",6.99,"309,827.04",True
Livros,"108,335.80",2.45,"108,335.80",True


In [17]:
# Consulta 7 do arquivo SQL: categoria líder em cada região, usando função de janela RANK()
pd.read_sql_query("""
    SELECT regiao, categoria, ROUND(receita_total, 2) AS receita_total FROM (
        SELECT regiao, categoria, SUM(receita) AS receita_total,
               RANK() OVER (PARTITION BY regiao ORDER BY SUM(receita) DESC) AS posicao
        FROM vendas
        GROUP BY regiao, categoria
    )
    WHERE posicao = 1
    ORDER BY receita_total DESC
""", conexao)

,regiao,categoria,receita_total
0,Sudeste,Eletrônicos,"1,011,895.71"
1,Nordeste,Eletrônicos,"482,664.94"
2,Sul,Eletrônicos,"366,107.72"
3,Centro-Oeste,Eletrônicos,"201,030.40"
4,Norte,Eletrônicos,"116,350.24"


In [18]:
conexao.close()

## 7. Indicadores e conclusões

In [19]:
indicadores = pd.Series({
    "Receita total (R$)": f"{metricas['receita_total']:,.2f}",
    "Número de vendas": f"{metricas['total_vendas']:,}",
    "Ticket médio (R$)": f"{metricas['receita_total'] / metricas['total_vendas']:,.2f}",
    "Crescimento 2024 x 2023": f"{metricas['crescimento_yoy_pct']:.1f}%",
    "Categoria líder": f"{metricas['categoria_top']} ({metricas['categoria_top_pct']:.1f}%)",
    "Região líder": metricas["regiao_top"],
    "% Online em 2024": f"{metricas['pct_online_ultimo_ano']:.1f}%",
    "Efeito Nov/Dez": f"{efeito:.2f}x",
}, name="valor")
indicadores.to_frame()

,valor
Receita total (R$),"4,429,491.93"
Número de vendas,"6,655"
Ticket médio (R$),665.59
Crescimento 2024 x 2023,19.8%
Categoria líder,Eletrônicos (49.2%)
Região líder,Sudeste
% Online em 2024,62.6%
Efeito Nov/Dez,1.26x


**Conclusões** (sobre os dados sintéticos)

1. **O crescimento veio de volume, não de preço.** A receita subiu 19,8% em 2024, puxada por 16,4% mais vendas; o ticket médio subiu só 2,9%. Ações para ganhar mais clientes e mais pedidos tiveram mais efeito que aumentos de preço.
2. **Eletrônicos é o centro do negócio, por preço e não por frequência.** Todas as categorias vendem quase o mesmo número de vezes, mas o ticket de Eletrônicos (R$ 1.644) faz a categoria somar ~49% da receita e ocupar os 4 primeiros produtos do ranking. É também a que mais cresceu (+23,5%). Isso é força e risco ao mesmo tempo: estoque e preço de eletrônicos merecem prioridade.
3. **O Online é o motor do crescimento.** A participação subiu de 55,4% para 62,6% e a loja física ficou estável, então quase todo o ganho de 2024 veio do canal digital.
4. **A sazonalidade existe, mas é moderada.** Nov/Dez vendem 1,26x a média dos outros meses (dezembro é o pico; abril e fevereiro, os vales). Planejar estoque com um pico de ~25% é mais realista que supor que o fim de ano dobra as vendas.
5. **SQL e pandas chegam aos mesmos números**, o que dá segurança de que consultas, notebook e dashboard do Power BI descrevem os mesmos dados.

**Limitações**: os dados são sintéticos (os padrões refletem as regras do gerador); o preço depende só da categoria, não do produto; não há custos, margem, clientes ou devoluções; e são apenas dois anos de histórico. Detalhes no README.